In [1]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, ParameterSampler

import joblib

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False


In [ ]:

import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

LABEL = "fraud"
IDCOL = "id"

ARTIFACT_DIR = Path("../../DATA/artifacts/stage1_models")
OUT_DIR = Path("../../DATA/stage1_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# - APPROVE: 정상 승인
# - REVIEW: 의심 거래(심사/추가인증/콜백 등) -> Stage2 대상
# - DECLINE: 부정거래 차단(거절/승인거절)
DECISION_APPROVE = "APPROVE"
DECISION_REVIEW  = "REVIEW"
DECISION_DECLINE = "DECLINE"


def _ensure_binary(y):
    y = np.asarray(y).astype(int)
    return y


def _fit_predict_score(model, X_tr, y_tr, X_te):
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    if hasattr(model, "predict_proba"):
        s_te = model.predict_proba(X_te)[:, 1]
    else:
        z = model.decision_function(X_te)
        s_te = 1.0 / (1.0 + np.exp(-z))
    pred_s = time.perf_counter() - t1

    return s_te.astype(float), fit_s, pred_s


def quantile_threshold(score, q):
    score = np.asarray(score).astype(float)
    return float(np.quantile(score, q))


def bucketize_3way(score, thr_review, thr_decline):
    """
    score >= thr_decline  -> DECLINE (차단)
    score >= thr_review   -> REVIEW  (심사/추가인증)
    else                  -> APPROVE (정상 승인)
    """
    score = np.asarray(score).astype(float)
    decision = np.full(score.shape[0], DECISION_APPROVE, dtype=object)
    decision[score >= thr_review] = DECISION_REVIEW
    decision[score >= thr_decline] = DECISION_DECLINE
    return decision


def report_binary(y_true, y_pred, title):
    print("=" * 80)
    print(title)
    print(classification_report(y_true, y_pred, digits=4))


def save_review_parquet(df_src, decision, path, keep_cols=None):
    if keep_cols is None:
        keep_cols = [IDCOL, LABEL]
    keep_cols = [c for c in keep_cols if c in df_src.columns]

    review_mask = (decision == DECISION_REVIEW)
    df_review = df_src.loc[review_mask, keep_cols].copy()

    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df_review.to_parquet(path, index=False)

    return df_review.shape[0]



def build_models(random_state=42):
    models = {}

    # 1) Logistic Regression 
    models["logit"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            n_jobs=None,
            class_weight=None,
            random_state=random_state,
        ))
    ])

    # 2) Calibrated Logistic 
    models["logit_calib"] = CalibratedClassifierCV(
        estimator=Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False)),
            ("clf", LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                class_weight=None,
                random_state=random_state,
            ))
        ]),
        method="sigmoid",
        cv=3,
    )

    # 3) HistGradientBoosting 
    models["hgb"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(
            loss="log_loss",
            max_depth=6,
            learning_rate=0.05,
            max_iter=300,
            random_state=random_state,
        ))
    ])

    # 4) Naive Bayes 
    models["gnb"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GaussianNB())
    ])

    # 5) LightGBM 
    if HAS_LGB:
        models["lgb_small"] = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            min_data_in_leaf=200,
            subsample=1.0,
            colsample_bytree=1.0,
            reg_lambda=0.0,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        )

    return models



def run_stage1_models(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    feature_cols,
    label_col=LABEL,
    id_col=IDCOL,
    review_q=0.99,        
    decline_q=0.999,      
    review_out_path="../../DATA/dataset/STAGE2_REVIEW_IDS.parquet",
):

    X_tr = df_train[feature_cols]
    y_tr = _ensure_binary(df_train[label_col])

    X_te = df_test[feature_cols]
    y_te = _ensure_binary(df_test[label_col])

    models = build_models()

    results = []

    for name in tqdm(list(models.keys()), desc="Stage1 model sweep"):
        model = models[name]

        score_te, fit_s, pred_s = _fit_predict_score(model, X_tr, y_tr, X_te)

        thr_review = quantile_threshold(score_te, review_q)
        thr_decline = quantile_threshold(score_te, decline_q)

        decision = bucketize_3way(score_te, thr_review, thr_decline)

        y_pred_flag = ((decision == DECISION_REVIEW) | (decision == DECISION_DECLINE)).astype(int)

        title = (
            f"[{name}] flagged=(REVIEW|DECLINE) "
            f"review_q={review_q} decline_q={decline_q} "
            f"thr_review={thr_review:.6f} thr_decline={thr_decline:.6f} "
            f"fit={fit_s:.3f}s pred={pred_s:.3f}s"
        )
        report_binary(y_te, y_pred_flag, title)


        n_app = int((decision == DECISION_APPROVE).sum())
        n_rev = int((decision == DECISION_REVIEW).sum())
        n_dec = int((decision == DECISION_DECLINE).sum())

        review_path = OUT_DIR / f"REVIEW_{name}.parquet"
        n_saved = save_review_parquet(
            df_src=df_test[[id_col, label_col]].copy(),
            decision=decision,
            path=review_path,
            keep_cols=[id_col, label_col],
        )

        results.append({
            "model": name,
            "fit_sec": fit_s,
            "pred_sec": pred_s,
            "thr_review": thr_review,
            "thr_decline": thr_decline,
            "n_approve": n_app,
            "n_review": n_rev,
            "n_decline": n_dec,
            "review_saved": n_saved,
            "review_path": str(review_path),
        })

    return pd.DataFrame(results).sort_values(["fit_sec", "pred_sec"], ascending=True).reset_index(drop=True)


In [8]:
df_train = pd.read_parquet("../../DATA/dataset/TRAIN_stage1")
df_test  = pd.read_parquet("../../DATA/dataset/TEST_stage1")

feature_cols = [c for c in df_train.columns if c not in [IDCOL, LABEL]]
summary = run_stage1_models(
    df_train, df_test,
    feature_cols=feature_cols,
    review_q=0.99,
    decline_q=0.999,
)
display(summary)
summary.to_csv(OUT_DIR / "stage1_model_sweep_summary.csv", index=False)

Stage1 model sweep:   0%|          | 0/5 [00:00<?, ?it/s]

[logit] flagged=(REVIEW|DECLINE) review_q=0.99 decline_q=0.999 thr_review=0.088086 thr_decline=0.998699 fit=2.116s pred=0.037s
              precision    recall  f1-score   support

           0     0.9899    0.9983    0.9941    112113
           1     0.8338    0.4547    0.5885      2096

    accuracy                         0.9883    114209
   macro avg     0.9118    0.7265    0.7913    114209
weighted avg     0.9870    0.9883    0.9866    114209

[logit_calib] flagged=(REVIEW|DECLINE) review_q=0.99 decline_q=0.999 thr_review=0.084488 thr_decline=0.997467 fit=7.267s pred=0.190s
              precision    recall  f1-score   support

           0     0.9899    0.9983    0.9941    112113
           1     0.8346    0.4552    0.5891      2096

    accuracy                         0.9883    114209
   macro avg     0.9123    0.7267    0.7916    114209
weighted avg     0.9871    0.9883    0.9867    114209

[hgb] flagged=(REVIEW|DECLINE) review_q=0.99 decline_q=0.999 thr_review=0.980971 thr_d

,model,fit_sec,pred_sec,thr_review,thr_decline,n_approve,n_review,n_decline,review_saved,review_path
0,gnb,0.496136,0.026828,1.000000,1.000000,113021,0,1188,0,../../DATA/stage1_outputs/REVIEW_gnb.parquet
1,lgb_small,1.662906,0.040484,0.992649,0.999858,113066,1028,115,1028,../../DATA/stage1_outputs/REVIEW_lgb_small.par...
2,logit,2.115514,0.036531,0.088086,0.998699,113066,1028,115,1028,../../DATA/stage1_outputs/REVIEW_logit.parquet
3,hgb,2.758344,0.113618,0.980971,0.999764,113066,1028,115,1028,../../DATA/stage1_outputs/REVIEW_hgb.parquet
4,logit_calib,7.266914,0.189929,0.084488,0.997467,113066,1028,115,1028,../../DATA/stage1_outputs/REVIEW_logit_calib.p...


## Stage1 성능 판단 기준 및 시간 지표

* 운영 제약: `review_q=0.99`, `decline_q=0.999`로 **심사(Review) 1% + 차단(Decline) 0.1%**의 고정된 처리량 하에서 모델 비교
* Stage1에서 중요한 성능 척도

  * `precision (class=1)`: Stage1이 올리는 알람(심사/차단)의 **정확도**. 높을수록 불필요한 심사/차단(고객 불편, 운영비용)이 줄어듦
  * `recall (class=1)`: 전체 부정거래 중 Stage1이 **얼마나 많이 건지는지(검거율)**. 낮으면 Stage2로 넘어가지 못하고 승인 구간으로 빠지는 부정거래가 증가
  * `F1 (class=1)`: precision–recall 균형 지표. 운영량이 고정된 상황에서 모델 비교 시 참고 지표로 유효
* 시간 지표(Stage1 적합성)

  * `fit_sec`: 오프라인 재학습/리프레시 비용
  * `pred_sec`: 온라인 추론 비용(실시간 처리). Stage1은 online latency에 민감하므로 예측 시간이 핵심

---

## 모델별 Classification Report 요약 (class=1 중심)

| model       | precision (1) | recall (1) | f1 (1) | fit_sec | pred_sec |
| ----------- | ------------: | ---------: | -----: | ------: | -------: |
| logit       |        0.8338 |     0.4547 | 0.5885 |   2.116 |    0.037 |
| logit_calib |        0.8346 |     0.4552 | 0.5891 |   7.267 |    0.190 |
| hgb         |        0.8215 |     0.4480 | 0.5798 |   2.758 |    0.114 |
| gnb         |        0.5168 |     0.2929 | 0.3739 |   0.496 |    0.027 |
| lgb_small   |        0.8093 |     0.4413 | 0.5712 |   1.663 |    0.040 |

---

## Stage1 모델을 logit으로 선택한 이유

* 동일 처리량(심사 1% + 차단 0.1%) 조건에서 **부정거래 검거율(recall)이 가장 높음**

  * logit(0.4547) ≥ logit_calib(0.4552, 거의 동일) > hgb(0.4480) > lgb_small(0.4413) >> gnb(0.2929)
  * Stage1은 “다음 단계(Stage2)로 넘길 부정 의심 건을 최대한 확보”하는 역할이므로 recall 우선순위가 높음
* 동시에 **정밀도(precision)도 최상위권**

  * logit precision(0.8338)로, 운영량이 고정된 상황에서 불필요한 심사/차단 비중을 낮게 유지
* **추론 시간(pred_sec)이 매우 짧아 Stage1 온라인 요구에 적합**

  * logit pred 0.037s로 빠른 편이며, lgb_small과 유사 수준
* logit_calib는 성능(precision/recall/F1)이 logit과 **거의 동일하지만 비용이 크게 증가**

  * fit 7.267s, pred 0.190s로 Stage1에서 불리
  * 확률 보정(calibration)의 이점이 이 조건에서는 성능으로 이어지지 않았음
* 트리 계열(hgb, lgb_small)은 현 피처셋/운영량 조건에서 logit 대비 **recall/precision이 동시에 열세**

  * Stage1의 목적(빠르고 안정적인 1차 필터) 관점에서 logit 대비 채택 근거가 약함
* gnb는 class=1 성능이 낮고(threshold가 1로 몰리는 등) 운영 안정성이 떨어져 Stage1에 부적합

---

## 결론

* Stage1 목표(온라인 추론 효율 + 제한된 알람 처리량 하에서 최대 검거)를 기준으로 보면, 현재 실험 조건에서는 **logit이 성능(precision/recall/F1)과 시간(pred_sec) 균형이 가장 좋음**
* 따라서 Stage1 운영 모델은 **logit**을 채택하는 것이 합리적임


In [9]:
# Stage1 Logit 튜닝: (C, penalty, class_weight) sweep + 운영량(Review/Decline) 고정(Top-K) + classification_report + 시간 측정

import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


LABEL = "fraud"
IDCOL = "id"

DECISION_APPROVE = "APPROVE"
DECISION_REVIEW  = "REVIEW"
DECISION_DECLINE = "DECLINE"


def _ensure_int(y):
    return np.asarray(y).astype(int)


def topk_bucketize_3way(score, review_rate=0.01, decline_rate=0.001):
    score = np.asarray(score).astype(float)
    n = score.shape[0]

    k_review_total = int(np.floor(n * float(review_rate)))
    k_decline = int(np.floor(n * float(decline_rate)))
    k_decline = min(k_decline, k_review_total)

    order = np.argsort(-score)  # desc

    decision = np.full(n, DECISION_APPROVE, dtype=object)

    if k_review_total > 0:
        idx_review_total = order[:k_review_total]
        decision[idx_review_total] = DECISION_REVIEW

    if k_decline > 0:
        idx_decline = order[:k_decline]
        decision[idx_decline] = DECISION_DECLINE

    # 반환: decision, (thr_review, thr_decline) 참고용
    thr_review = float(score[order[k_review_total - 1]]) if k_review_total > 0 else None
    thr_decline = float(score[order[k_decline - 1]]) if k_decline > 0 else None
    return decision, thr_review, thr_decline


def flagged_pred_from_decision(decision):
    return ((decision == DECISION_REVIEW) | (decision == DECISION_DECLINE)).astype(int)


def make_logit_pipeline(C=1.0, penalty="l2", class_weight=None, random_state=42):
    if penalty == "l1":
        solver = "liblinear"
    else:
        solver = "lbfgs"

    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            C=float(C),
            penalty=penalty,
            solver=solver,
            max_iter=3000,
            class_weight=class_weight,
            random_state=random_state,
        )),
    ])


def save_review_ids(df_test, decision, out_path, keep_cols=None):
    if keep_cols is None:
        keep_cols = [IDCOL]
    keep_cols = [c for c in keep_cols if c in df_test.columns]

    mask = (decision == DECISION_REVIEW)
    df_out = df_test.loc[mask, keep_cols].copy()

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_parquet(out_path, index=False)
    return int(mask.sum())


def tune_stage1_logit(
    df_train,
    df_test,
    feature_cols,
    review_rate=0.01,
    decline_rate=0.001,
    C_grid=(0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0),
    penalty_grid=("l2", "l1"),
    class_weight_grid=(None, "balanced"),
    out_dir="../../DATA/stage1_outputs/logit_tuning",
):
    X_tr = df_train[feature_cols]
    y_tr = _ensure_int(df_train[LABEL])

    X_te = df_test[feature_cols]
    y_te = _ensure_int(df_test[LABEL])

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    combos = [
        (C, pen, cw)
        for C in C_grid
        for pen in penalty_grid
        for cw in class_weight_grid
    ]

    for C, pen, cw in tqdm(combos, desc="Logit tuning sweep"):
        model = make_logit_pipeline(C=C, penalty=pen, class_weight=cw)

        t0 = time.perf_counter()
        model.fit(X_tr, y_tr)
        fit_s = time.perf_counter() - t0

        t1 = time.perf_counter()
        score_te = model.predict_proba(X_te)[:, 1].astype(float)
        pred_s = time.perf_counter() - t1

        decision, thr_review, thr_decline = topk_bucketize_3way(
            score_te, review_rate=review_rate, decline_rate=decline_rate
        )
        y_pred_flag = flagged_pred_from_decision(decision)

        title = (
            f"[logit] C={C} penalty={pen} class_weight={cw} "
            f"review_rate={review_rate} decline_rate={decline_rate} "
            f"thr_review={None if thr_review is None else f'{thr_review:.6f}'} "
            f"thr_decline={None if thr_decline is None else f'{thr_decline:.6f}'} "
            f"fit={fit_s:.3f}s pred={pred_s:.3f}s"
        )

        print("=" * 80)
        print(title)
        print(classification_report(y_te, y_pred_flag, digits=4))

        n_app = int((decision == DECISION_APPROVE).sum())
        n_rev = int((decision == DECISION_REVIEW).sum())
        n_dec = int((decision == DECISION_DECLINE).sum())

        review_path = out_dir / f"REVIEW_C{C}_pen{pen}_cw{cw}.parquet"
        n_saved = save_review_ids(df_test[[IDCOL, LABEL]].copy(), decision, review_path, keep_cols=[IDCOL])

        rep = classification_report(y_te, y_pred_flag, output_dict=True, zero_division=0)

        rows.append({
            "C": float(C),
            "penalty": pen,
            "class_weight": str(cw),
            "fit_sec": float(fit_s),
            "pred_sec": float(pred_s),
            "thr_review": None if thr_review is None else float(thr_review),
            "thr_decline": None if thr_decline is None else float(thr_decline),
            "n_approve": n_app,
            "n_review": n_rev,
            "n_decline": n_dec,
            "precision_1": float(rep["1"]["precision"]),
            "recall_1": float(rep["1"]["recall"]),
            "f1_1": float(rep["1"]["f1-score"]),
            "support_1": int(rep["1"]["support"]),
            "review_saved": int(n_saved),
            "review_path": str(review_path),
        })

    df_res = pd.DataFrame(rows).sort_values(
        ["f1_1", "recall_1", "precision_1", "pred_sec"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

    df_res.to_csv(out_dir / "logit_tuning_summary.csv", index=False)
    return df_res

In [10]:
train = pd.read_parquet("../../DATA/dataset/TRAIN_stage1")
test  = pd.read_parquet("../../DATA/dataset/TEST_stage1")
feats = [c for c in train.columns if c not in [IDCOL, LABEL]]
res = tune_stage1_logit(train, test, feats, review_rate=0.01, decline_rate=0.001)
res.head(10)

Logit tuning sweep:   0%|          | 0/28 [00:00<?, ?it/s]

[logit] C=0.01 penalty=l2 class_weight=None review_rate=0.01 decline_rate=0.001 thr_review=0.086177 thr_decline=0.998314 fit=1.617s pred=0.033s
              precision    recall  f1-score   support

           0     0.9899    0.9983    0.9941    112113
           1     0.8363    0.4556    0.5899      2096

    accuracy                         0.9884    114209
   macro avg     0.9131    0.7270    0.7920    114209
weighted avg     0.9871    0.9884    0.9867    114209

[logit] C=0.01 penalty=l2 class_weight=balanced review_rate=0.01 decline_rate=0.001 thr_review=0.997100 thr_decline=1.000000 fit=2.253s pred=0.033s
              precision    recall  f1-score   support

           0     0.9905    0.9989    0.9947    112113
           1     0.8932    0.4866    0.6300      2096

    accuracy                         0.9895    114209
   macro avg     0.9418    0.7428    0.8123    114209
weighted avg     0.9887    0.9895    0.9880    114209

[logit] C=0.01 penalty=l1 class_weight=None review_rat

,C,penalty,class_weight,fit_sec,pred_sec,thr_review,thr_decline,n_approve,n_review,n_decline,precision_1,recall_1,f1_1,support_1,review_saved,review_path
0,0.30,l1,balanced,5.396824,0.013708,0.997800,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...
1,0.01,l1,balanced,5.181197,0.013731,0.997563,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...
2,1.00,l1,balanced,5.503360,0.013739,0.997804,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...
3,3.00,l1,balanced,5.401984,0.013772,0.997805,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...
4,0.30,l2,balanced,2.710863,0.017746,0.997787,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...
5,10.00,l2,balanced,2.624323,0.017811,0.997806,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...
6,0.10,l1,balanced,5.505215,0.021707,0.997784,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...
7,1.00,l2,balanced,2.568791,0.024983,0.997801,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...
8,3.00,l2,balanced,2.484463,0.025053,0.997805,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...
9,0.03,l1,balanced,4.988455,0.027619,0.997728,1.0,113067,1028,114,0.89317,0.486641,0.630019,2096,1028,../../DATA/stage1_outputs/logit_tuning/REVIEW_...


In [ ]:
LABEL = "fraud"
IDCOL = "id"

def print_best_logit_report(df_train, df_test, feature_cols, df_tuning_summary,
                            review_rate=0.01, decline_rate=0.001):

    best = (
        df_tuning_summary
        .sort_values(["f1_1", "recall_1", "precision_1", "pred_sec"],
                     ascending=[False, False, False, True])
        .iloc[0]
    )

    C = float(best["C"])
    penalty = str(best["penalty"])
    cw = None if best["class_weight"] in ["None", "none", "nan"] else str(best["class_weight"])

    X_tr = df_train[feature_cols]
    y_tr = df_train[LABEL].astype(int).values

    X_te = df_test[feature_cols]
    y_te = df_test[LABEL].astype(int).values

    model = make_logit_pipeline(C=C, penalty=penalty, class_weight=cw)

    model.fit(X_tr, y_tr)

    score_te = model.predict_proba(X_te)[:, 1].astype(float)

    decision, thr_review, thr_decline = topk_bucketize_3way(
        score_te, review_rate=review_rate, decline_rate=decline_rate
    )
    y_pred_flag = flagged_pred_from_decision(decision)

    print("=" * 80)
    print(f"[BEST LOGIT] C={C} penalty={penalty} class_weight={cw} "
          f"review_rate={review_rate} decline_rate={decline_rate} "
          f"thr_review={thr_review:.6f} thr_decline={thr_decline:.6f}")
    print(classification_report(y_te, y_pred_flag, digits=4))

    return best

In [12]:
train = pd.read_parquet("../../DATA/dataset/TRAIN_stage1")
test  = pd.read_parquet("../../DATA/dataset/TEST_stage1")
feats = [c for c in train.columns if c not in [IDCOL, LABEL]]

tuning = pd.read_csv("../../DATA/stage1_outputs/logit_tuning/logit_tuning_summary.csv")
best_row = print_best_logit_report(train, test, feats, tuning, review_rate=0.01, decline_rate=0.001)
best_row

[BEST LOGIT] C=0.3 penalty=l1 class_weight=balanced review_rate=0.01 decline_rate=0.001 thr_review=0.997800 thr_decline=1.000000
              precision    recall  f1-score   support

           0     0.9905    0.9989    0.9947    112113
           1     0.8932    0.4866    0.6300      2096

    accuracy                         0.9895    114209
   macro avg     0.9418    0.7428    0.8123    114209
weighted avg     0.9887    0.9895    0.9880    114209



C                                                             0.3
penalty                                                        l1
class_weight                                             balanced
fit_sec                                                  5.396824
pred_sec                                                 0.013708
thr_review                                                 0.9978
thr_decline                                                   1.0
n_approve                                                  113067
n_review                                                     1028
n_decline                                                     114
precision_1                                               0.89317
recall_1                                                 0.486641
f1_1                                                     0.630019
support_1                                                    2096
review_saved                                                 1028
review_pat

In [ ]:
# Stage1 -> Stage2 운영형 출력 생성기 (BEST logit 재학습 + 3단계 의사결정 + parquet 저장 + 리포트 출력)

import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report


LABEL = "fraud"
IDCOL = "id"

DECISION_APPROVE = "APPROVE"
DECISION_REVIEW  = "REVIEW"
DECISION_DECLINE = "DECLINE"


def _ensure_int(y):
    return np.asarray(y).astype(int)


def topk_bucketize_3way(score, review_rate=0.01, decline_rate=0.001):
    score = np.asarray(score).astype(float)
    n = score.shape[0]

    k_review_total = int(np.floor(n * float(review_rate)))
    k_decline = int(np.floor(n * float(decline_rate)))
    k_decline = min(k_decline, k_review_total)

    order = np.argsort(-score)  # desc

    decision = np.full(n, DECISION_APPROVE, dtype=object)

    if k_review_total > 0:
        idx_review_total = order[:k_review_total]
        decision[idx_review_total] = DECISION_REVIEW

    if k_decline > 0:
        idx_decline = order[:k_decline]
        decision[idx_decline] = DECISION_DECLINE

    thr_review = float(score[order[k_review_total - 1]]) if k_review_total > 0 else None
    thr_decline = float(score[order[k_decline - 1]]) if k_decline > 0 else None
    return decision, thr_review, thr_decline


def flagged_pred_from_decision(decision):
    return ((decision == DECISION_REVIEW) | (decision == DECISION_DECLINE)).astype(int)


def _save_ids(df, mask, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.loc[mask, [IDCOL]].copy().to_parquet(out_path, index=False)
    return int(mask.sum())


def print_and_export_best_stage1(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    feature_cols,
    df_tuning_summary: pd.DataFrame,
    review_rate=0.01,
    decline_rate=0.001,
    out_dir="../../DATA/stage1_outputs/final_best_logit",
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # best row 선택
    best = (
        df_tuning_summary
        .sort_values(["f1_1", "recall_1", "precision_1", "pred_sec"],
                     ascending=[False, False, False, True])
        .iloc[0]
    )

    C = float(best["C"])
    penalty = str(best["penalty"])
    cw = None if str(best["class_weight"]) in ["None", "none", "nan"] else str(best["class_weight"])

    # 학습/평가
    X_tr = df_train[feature_cols]
    y_tr = _ensure_int(df_train[LABEL])

    X_te = df_test[feature_cols]
    y_te = _ensure_int(df_test[LABEL])

    model = make_logit_pipeline(C=C, penalty=penalty, class_weight=cw)

    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    score_te = model.predict_proba(X_te)[:, 1].astype(float)
    pred_s = time.perf_counter() - t1

    decision, thr_review, thr_decline = topk_bucketize_3way(
        score_te, review_rate=review_rate, decline_rate=decline_rate
    )
    y_pred_flag = flagged_pred_from_decision(decision)

    print("=" * 80)
    print(f"[FINAL BEST LOGIT] C={C} penalty={penalty} class_weight={cw} "
          f"review_rate={review_rate} decline_rate={decline_rate} "
          f"thr_review={thr_review:.6f} thr_decline={thr_decline:.6f} "
          f"fit={fit_s:.3f}s pred={pred_s:.3f}s")
    print(classification_report(y_te, y_pred_flag, digits=4))

    # 저장
    m_app = (decision == DECISION_APPROVE)
    m_rev = (decision == DECISION_REVIEW)
    m_dec = (decision == DECISION_DECLINE)

    approve_path = out_dir / "APPROVE_ids.parquet"
    review_path  = out_dir / "REVIEW_ids_for_stage2.parquet"
    decline_path = out_dir / "DECLINE_ids.parquet"

    n_app = _save_ids(df_test, m_app, approve_path)
    n_rev = _save_ids(df_test, m_rev, review_path)
    n_dec = _save_ids(df_test, m_dec, decline_path)


    log_cols = [IDCOL]
    if LABEL in df_test.columns:
        log_cols.append(LABEL)

    df_log = df_test[log_cols].copy()
    df_log["stage1_score"] = score_te.astype("float32")
    df_log["stage1_decision"] = decision
    df_log.to_parquet(out_dir / "STAGE1_scoring_log.parquet", index=False)

    # 요약
    summary = {
        "C": C,
        "penalty": penalty,
        "class_weight": cw,
        "review_rate": float(review_rate),
        "decline_rate": float(decline_rate),
        "thr_review": float(thr_review),
        "thr_decline": float(thr_decline),
        "fit_sec": float(fit_s),
        "pred_sec": float(pred_s),
        "n_approve": int(n_app),
        "n_review": int(n_rev),
        "n_decline": int(n_dec),
        "approve_path": str(approve_path),
        "review_path": str(review_path),
        "decline_path": str(decline_path),
        "log_path": str(out_dir / "STAGE1_scoring_log.parquet"),
    }

    print("-" * 80)
    print("Saved:")
    print("APPROVE:", approve_path, "n=", n_app)
    print("REVIEW :", review_path,  "n=", n_rev, "(Stage2 input)")
    print("DECLINE:", decline_path, "n=", n_dec)
    print("LOG    :", out_dir / "STAGE1_scoring_log.parquet")

    return best, summary

In [16]:
train = pd.read_parquet("../../DATA/dataset/TRAIN_stage1")
test  = pd.read_parquet("../../DATA/dataset/TEST_stage1")
feats = [c for c in train.columns if c not in [IDCOL, LABEL]]

tuning = pd.read_csv("../../DATA/stage1_outputs/logit_tuning/logit_tuning_summary.csv")
best_row, summary = print_and_export_best_stage1(
    train, test, feats, tuning,
    review_rate=0.01, decline_rate=0.001,
    out_dir="../../DATA/stage1_outputs/final_best_logit",
)
summary

[FINAL BEST LOGIT] C=0.3 penalty=l1 class_weight=balanced review_rate=0.01 decline_rate=0.001 thr_review=0.997800 thr_decline=1.000000 fit=5.608s pred=0.028s
              precision    recall  f1-score   support

           0     0.9905    0.9989    0.9947    112113
           1     0.8932    0.4866    0.6300      2096

    accuracy                         0.9895    114209
   macro avg     0.9418    0.7428    0.8123    114209
weighted avg     0.9887    0.9895    0.9880    114209

--------------------------------------------------------------------------------
Saved:
APPROVE: ../../DATA/stage1_outputs/final_best_logit/APPROVE_ids.parquet n= 113067
REVIEW : ../../DATA/stage1_outputs/final_best_logit/REVIEW_ids_for_stage2.parquet n= 1028 (Stage2 input)
DECLINE: ../../DATA/stage1_outputs/final_best_logit/DECLINE_ids.parquet n= 114
LOG    : ../../DATA/stage1_outputs/final_best_logit/STAGE1_scoring_log.parquet


{'C': 0.3,
 'penalty': 'l1',
 'class_weight': 'balanced',
 'review_rate': 0.01,
 'decline_rate': 0.001,
 'thr_review': 0.9978000714811825,
 'thr_decline': 1.0,
 'fit_sec': 5.608391575049609,
 'pred_sec': 0.028141759801656008,
 'n_approve': 113067,
 'n_review': 1028,
 'n_decline': 114,
 'approve_path': '../../DATA/stage1_outputs/final_best_logit/APPROVE_ids.parquet',
 'review_path': '../../DATA/stage1_outputs/final_best_logit/REVIEW_ids_for_stage2.parquet',
 'decline_path': '../../DATA/stage1_outputs/final_best_logit/DECLINE_ids.parquet',
 'log_path': '../../DATA/stage1_outputs/final_best_logit/STAGE1_scoring_log.parquet'}